In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables from .env
load_dotenv()

# Initialize the LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",                   # correct parameter
    google_api_key=os.environ["GOOGLE_API_KEY"],  # correct parameter
    temperature=0
)

# Send a message
response = llm.invoke("Hi")
print(response.content)

Hi there! How can I help you today?


In [5]:
class JokeState(TypedDict):
    topic : str
    joke  : str
    explanation : str

In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded** the dough!',
 'explanation': 'This joke is a classic **pun**, playing on two words that sound alike but have different meanings:\n\n1.  **"Kneaded" (pronounced "need-ed"):** This is what you do to **dough** when you\'re making pizza (or bread). You work and press it with your hands.\n2.  **"Needed" (pronounced "need-ed"):** This means "required" or "wanted."\n\nAnd there\'s a second layer of wordplay with "dough":\n\n*   **Dough (for pizza):** The actual mixture you knead.\n*   **Dough (slang):** A common slang term for **money**.\n\n**Here\'s how the joke works:**\n\n*   **Why did the pizza get a job?** People get jobs to earn money.\n*   **Because it kneaded the dough!**\n    *   Literally, a pizza (or the person making it) would "knead the dough" as part of its creation.\n    *   But the joke implies it "needed the dough" (money) from a job.\n\nThe humor comes from the clever way it uses the same sou

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it **kneaded** the dough!', 'explanation': 'This joke is a classic **pun**, playing on two words that sound alike but have different meanings:\n\n1.  **"Kneaded" (pronounced "need-ed"):** This is what you do to **dough** when you\'re making pizza (or bread). You work and press it with your hands.\n2.  **"Needed" (pronounced "need-ed"):** This means "required" or "wanted."\n\nAnd there\'s a second layer of wordplay with "dough":\n\n*   **Dough (for pizza):** The actual mixture you knead.\n*   **Dough (slang):** A common slang term for **money**.\n\n**Here\'s how the joke works:**\n\n*   **Why did the pizza get a job?** People get jobs to earn money.\n*   **Because it kneaded the dough!**\n    *   Literally, a pizza (or the person making it) would "knead the dough" as part of its creation.\n    *   But the joke implies it "needed the dough" (money) from a job.\n\nThe humor comes from the clever way i